# World Weather Online Historical Weather Ingestion

Pull historical weather data for NFL stadiums to build weather similarity models.

**Features:**
- Free tier: 500 API calls/day
- Historical weather data (past games)
- Current weather and 14-day forecast (paid)
- Hourly historical data
- Wind speed, gusts, temperature, precipitation

**Use Case: Historical Weather Similarity Search**
- Find past games with similar weather conditions
- Analyze player performance in comparable weather
- Build weather-adjusted projections based on historical patterns
- Example: "Find all games with 20mph winds, 35°F, similar to today's forecast"

**Resources:**
- Website: https://www.worldweatheronline.com
- API Docs: https://www.worldweatheronline.com/developer/api/
- Free tier: 500 calls/day
- Sign up: https://www.worldweatheronline.com/developer/api/pricing.aspx

In [0]:
import requests
import pandas as pd
import json
from datetime import datetime, timedelta
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Configuration
BASE_URL = "https://api.worldweatheronline.com/premium/v1"
SEASON = 2024

# Get API key from Databricks Secrets
try:
    API_KEY = dbutils.secrets.get(scope="api-keys", key="worldweatheronline-key")
    print("✓ API key loaded from secrets")
except:
    print("⚠️ ERROR: API key not found in secrets")
    print("\nTo set up:")
    print("1. Sign up at https://www.worldweatheronline.com/developer/api/pricing.aspx")
    print("2. Choose Free tier (500 calls/day)")
    print("3. Get your API key from dashboard")
    print("4. Add to secrets: databricks secrets put --scope api-keys --key worldweatheronline-key")
    print("5. Or set manually: API_KEY = 'your-key-here'")
    # Uncomment to set manually
    # API_KEY = "your-worldweatheronline-key-here"
    raise

print(f"\n🌧️ World Weather Online Historical Weather Ingestion")
print(f"Season: {SEASON}")
print(f"API Endpoint: {BASE_URL}")
print(f"Free Tier: 500 calls/day")
print(f"\nUse Case: Historical weather similarity search")

In [0]:
# NFL stadium cities for historical weather lookup

NFL_STADIUMS_HISTORICAL = {
    'ARI': {'city': 'Glendale', 'state': 'Arizona', 'country': 'USA', 'is_dome': True},
    'ATL': {'city': 'Atlanta', 'state': 'Georgia', 'country': 'USA', 'is_dome': True},
    'BAL': {'city': 'Baltimore', 'state': 'Maryland', 'country': 'USA', 'is_dome': False},
    'BUF': {'city': 'Buffalo', 'state': 'New York', 'country': 'USA', 'is_dome': False},
    'CAR': {'city': 'Charlotte', 'state': 'North Carolina', 'country': 'USA', 'is_dome': False},
    'CHI': {'city': 'Chicago', 'state': 'Illinois', 'country': 'USA', 'is_dome': False},
    'CIN': {'city': 'Cincinnati', 'state': 'Ohio', 'country': 'USA', 'is_dome': False},
    'CLE': {'city': 'Cleveland', 'state': 'Ohio', 'country': 'USA', 'is_dome': False},
    'DAL': {'city': 'Arlington', 'state': 'Texas', 'country': 'USA', 'is_dome': True},
    'DEN': {'city': 'Denver', 'state': 'Colorado', 'country': 'USA', 'is_dome': False},
    'DET': {'city': 'Detroit', 'state': 'Michigan', 'country': 'USA', 'is_dome': True},
    'GB': {'city': 'Green Bay', 'state': 'Wisconsin', 'country': 'USA', 'is_dome': False},
    'HOU': {'city': 'Houston', 'state': 'Texas', 'country': 'USA', 'is_dome': True},
    'IND': {'city': 'Indianapolis', 'state': 'Indiana', 'country': 'USA', 'is_dome': True},
    'JAX': {'city': 'Jacksonville', 'state': 'Florida', 'country': 'USA', 'is_dome': False},
    'KC': {'city': 'Kansas City', 'state': 'Missouri', 'country': 'USA', 'is_dome': False},
    'LAC': {'city': 'Los Angeles', 'state': 'California', 'country': 'USA', 'is_dome': False},
    'LAR': {'city': 'Los Angeles', 'state': 'California', 'country': 'USA', 'is_dome': False},
    'LV': {'city': 'Las Vegas', 'state': 'Nevada', 'country': 'USA', 'is_dome': True},
    'MIA': {'city': 'Miami', 'state': 'Florida', 'country': 'USA', 'is_dome': False},
    'MIN': {'city': 'Minneapolis', 'state': 'Minnesota', 'country': 'USA', 'is_dome': True},
    'NE': {'city': 'Foxborough', 'state': 'Massachusetts', 'country': 'USA', 'is_dome': False},
    'NO': {'city': 'New Orleans', 'state': 'Louisiana', 'country': 'USA', 'is_dome': True},
    'NYG': {'city': 'New York', 'state': 'New York', 'country': 'USA', 'is_dome': False},
    'NYJ': {'city': 'New York', 'state': 'New York', 'country': 'USA', 'is_dome': False},
    'PHI': {'city': 'Philadelphia', 'state': 'Pennsylvania', 'country': 'USA', 'is_dome': False},
    'PIT': {'city': 'Pittsburgh', 'state': 'Pennsylvania', 'country': 'USA', 'is_dome': False},
    'SEA': {'city': 'Seattle', 'state': 'Washington', 'country': 'USA', 'is_dome': False},
    'SF': {'city': 'San Francisco', 'state': 'California', 'country': 'USA', 'is_dome': False},
    'TB': {'city': 'Tampa', 'state': 'Florida', 'country': 'USA', 'is_dome': False},
    'TEN': {'city': 'Nashville', 'state': 'Tennessee', 'country': 'USA', 'is_dome': False},
    'WAS': {'city': 'Washington', 'state': 'District of Columbia', 'country': 'USA', 'is_dome': False}
}

print(f"Loaded {len(NFL_STADIUMS_HISTORICAL)} NFL stadium locations")
outdoor = sum(1 for s in NFL_STADIUMS_HISTORICAL.values() if not s['is_dome'])
print(f"Outdoor stadiums (weather relevant): {outdoor}")

In [0]:
# Fetch historical weather for past NFL game days
# Example: Get weather from Week 18 games in 2024

import time

print("Fetching historical weather data...\n")
print("Sample: Fetching weather for 5 cold-weather stadiums from past game days\n")

historical_weather_data = []

# Sample teams and dates (typically Sunday games)
sample_teams = ['BUF', 'GB', 'CHI', 'NE', 'PIT']
sample_dates = [
    '2024-12-29',  # Week 17 Sunday
    '2024-12-22',  # Week 16 Sunday
    '2024-12-15',  # Week 15 Sunday
]

for team in sample_teams:
    if team not in NFL_STADIUMS_HISTORICAL:
        continue
        
    stadium = NFL_STADIUMS_HISTORICAL[team]
    city = stadium['city']
    
    for game_date in sample_dates:
        try:
            # Historical weather endpoint
            url = f"{BASE_URL}/past-weather.ashx"
            params = {
                'key': API_KEY,
                'q': city,
                'format': 'json',
                'date': game_date,
                'tp': '3'  # 3-hour intervals
            }
            
            response = requests.get(url, params=params, timeout=15)
            response.raise_for_status()
            
            data = response.json()
            
            # Extract weather data
            weather_data = data.get('data', {}).get('weather', [])
            
            if weather_data:
                day_weather = weather_data[0]
                hourly_data = day_weather.get('hourly', [])
                
                # Focus on game time (1PM ET typical, 13:00)
                # Get closest hourly reading to 1PM
                game_time_weather = None
                for hourly in hourly_data:
                    hour = int(hourly.get('time', '0')) / 100
                    if 12 <= hour <= 16:  # Game window
                        game_time_weather = hourly
                        break
                
                if not game_time_weather and hourly_data:
                    game_time_weather = hourly_data[len(hourly_data)//2]  # Midday
                
                if game_time_weather:
                    weather_record = {
                        'team': team,
                        'city': city,
                        'is_dome': stadium['is_dome'],
                        'game_date': game_date,
                        'temp_f': float(game_time_weather.get('tempF', 0)),
                        'feels_like_f': float(game_time_weather.get('FeelsLikeF', 0)),
                        'wind_speed_mph': float(game_time_weather.get('windspeedMiles', 0)),
                        'wind_gust_mph': float(game_time_weather.get('WindGustMiles', 0)),
                        'wind_dir': game_time_weather.get('winddir16Point', ''),
                        'wind_degree': int(game_time_weather.get('winddirDegree', 0)),
                        'humidity': int(game_time_weather.get('humidity', 0)),
                        'visibility_miles': float(game_time_weather.get('visibilityMiles', 0)),
                        'pressure_mb': float(game_time_weather.get('pressure', 0)),
                        'precip_in': float(game_time_weather.get('precipInches', 0)),
                        'cloud_cover': int(game_time_weather.get('cloudcover', 0)),
                        'condition': game_time_weather.get('weatherDesc', [{}])[0].get('value', '')
                    }
                    
                    historical_weather_data.append(weather_record)
                    
                    print(f"✓ {team} {game_date}: {weather_record['temp_f']}°F, "
                          f"Wind {weather_record['wind_speed_mph']}mph, "
                          f"{weather_record['condition']}")
            
            # Rate limit (500/day = ~20/hour = 1 every 3 seconds)
            time.sleep(3.2)
            
        except Exception as e:
            print(f"✗ {team} {game_date}: Error - {e}")

if historical_weather_data:
    historical_df = pd.DataFrame(historical_weather_data)
    print(f"\n✓ Fetched {len(historical_df)} historical weather records")
    display(historical_df[['team', 'game_date', 'temp_f', 'wind_speed_mph', 'wind_gust_mph', 'condition']].head(10))
else:
    print("\n⚠️ No historical data fetched")
    historical_df = pd.DataFrame()

In [0]:
# Build features for weather similarity matching

if 'historical_df' in locals() and len(historical_df) > 0:
    print("Building weather similarity features...\n")
    
    df = historical_df.copy()
    
    # Calculate impact scores (same as current weather)
    df['wind_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] else
        1 if row['wind_speed_mph'] < 10 else
        2 if row['wind_speed_mph'] < 15 else
        3 if row['wind_speed_mph'] < 20 else
        4,
        axis=1
    )
    
    df['cold_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] else
        1 if row['temp_f'] >= 40 else
        2 if row['temp_f'] >= 32 else
        3 if row['temp_f'] >= 20 else
        4,
        axis=1
    )
    
    df['precip_impact_score'] = df.apply(lambda row:
        0 if row['is_dome'] else
        1 if row['precip_in'] < 0.1 else
        2 if row['precip_in'] < 0.3 else
        3,
        axis=1
    )
    
    # Create weather signature for similarity matching
    df['weather_signature'] = (
        df['wind_impact_score'].astype(str) + '_' +
        df['cold_impact_score'].astype(str) + '_' +
        df['precip_impact_score'].astype(str)
    )
    
    # Example: Find games with similar weather
    print("Weather signatures (for similarity search):")
    print(df.groupby('weather_signature').size().sort_values(ascending=False))
    
    print("\nExample: Games with extreme conditions (signature 4_4_X or 4_X_4):")
    extreme_weather = df[
        (df['wind_impact_score'] >= 3) | 
        (df['cold_impact_score'] >= 3)
    ]
    if len(extreme_weather) > 0:
        display(extreme_weather[['team', 'game_date', 'temp_f', 'wind_speed_mph', 'weather_signature']])
    else:
        print("No extreme weather games in sample")
    
    historical_features_df = df
else:
    print("⚠️ No historical data to process")

In [0]:
# Transform historical weather to Spark DataFrame

if 'historical_features_df' in locals() and len(historical_features_df) > 0:
    print("Transforming to Spark DataFrame...\n")
    
    rows = []
    for idx, row in historical_features_df.iterrows():
        # Extract season/week from date (approximate)
        game_date = datetime.strptime(row['game_date'], '%Y-%m-%d')
        season = game_date.year if game_date.month >= 9 else game_date.year - 1
        
        # Estimate week (rough approximation)
        season_start = datetime(season, 9, 1)  # Approximate season start
        days_diff = (game_date - season_start).days
        week = max(1, min(18, days_diff // 7 + 1))
        
        spark_row = Row(
            team=row['team'],
            city=row['city'],
            is_dome=bool(row['is_dome']),
            game_date=row['game_date'],
            season=int(season),
            week=int(week),
            temp_f=float(row['temp_f']),
            feels_like_f=float(row['feels_like_f']),
            wind_speed_mph=float(row['wind_speed_mph']),
            wind_gust_mph=float(row['wind_gust_mph']),
            wind_dir=str(row['wind_dir']),
            wind_degree=int(row['wind_degree']),
            humidity=int(row['humidity']),
            visibility_miles=float(row['visibility_miles']),
            precip_in=float(row['precip_in']),
            condition=str(row['condition']),
            wind_impact_score=int(row['wind_impact_score']),
            cold_impact_score=int(row['cold_impact_score']),
            precip_impact_score=int(row['precip_impact_score']),
            weather_signature=str(row['weather_signature']),
            raw_data=json.dumps(row.to_dict(), default=str)
        )
        rows.append(spark_row)
    
    historical_spark_df = spark.createDataFrame(rows)
    print(f"✓ Created Spark DataFrame with {historical_spark_df.count()} records\n")
    display(historical_spark_df.limit(10))
else:
    print("⚠️ No data to transform")

In [0]:
# Write to bronze_nfl_weather_historical table

if 'historical_spark_df' in locals():
    print("Writing to bronze_nfl_weather_historical...\n")
    
    bronze_historical = historical_spark_df.withColumn("ingested_at", F.current_timestamp())
    bronze_historical = bronze_historical.withColumn("source", F.lit("worldweatheronline"))
    
    bronze_historical.createOrReplaceTempView("worldweatheronline_bronze_updates")
    
    # Create historical weather table
    spark.sql("""
        CREATE TABLE IF NOT EXISTS main.fantasai.bronze_nfl_weather_historical (
            team STRING,
            city STRING,
            is_dome BOOLEAN,
            game_date STRING,
            season INT,
            week INT,
            temp_f DOUBLE,
            feels_like_f DOUBLE,
            wind_speed_mph DOUBLE,
            wind_gust_mph DOUBLE,
            wind_dir STRING,
            wind_degree INT,
            humidity INT,
            visibility_miles DOUBLE,
            precip_in DOUBLE,
            condition STRING,
            wind_impact_score INT,
            cold_impact_score INT,
            precip_impact_score INT,
            weather_signature STRING,
            raw_data STRING,
            source STRING,
            ingested_at TIMESTAMP
        )
        USING DELTA
    """)
    
    # Merge data
    spark.sql("""
        MERGE INTO main.fantasai.bronze_nfl_weather_historical AS target
        USING worldweatheronline_bronze_updates AS source
        ON target.team = source.team 
            AND target.game_date = source.game_date
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    
    print(f"✓ Merged {bronze_historical.count()} historical weather records\n")
else:
    print("⚠️ No data to write")

In [0]:
%sql
-- Check historical weather data
SELECT 
    team,
    game_date,
    season,
    week,
    temp_f,
    wind_speed_mph,
    wind_gust_mph,
    condition,
    weather_signature
FROM main.fantasai.bronze_nfl_weather_historical
WHERE season = 2024
ORDER BY game_date DESC, wind_speed_mph DESC
LIMIT 20

In [0]:
%sql
-- Example: Find historical games with similar weather to a target game
-- Target: Cold (32°F), Windy (18mph) conditions

WITH target_conditions AS (
  SELECT 
    2 as target_cold_score,  -- 32°F = score 2
    3 as target_wind_score   -- 18mph = score 3
),
similar_games AS (
  SELECT 
    h.*,
    -- Calculate similarity (lower = more similar)
    ABS(h.cold_impact_score - t.target_cold_score) + 
    ABS(h.wind_impact_score - t.target_wind_score) as similarity_distance
  FROM main.fantasai.bronze_nfl_weather_historical h
  CROSS JOIN target_conditions t
  WHERE NOT h.is_dome
)
SELECT 
  team,
  game_date,
  temp_f,
  wind_speed_mph,
  wind_gust_mph,
  condition,
  weather_signature,
  similarity_distance
FROM similar_games
WHERE similarity_distance <= 1  -- Very similar
ORDER BY similarity_distance, game_date DESC
LIMIT 20

## World Weather Online Features

### Available Endpoints
1. **Past Weather** - `/past-weather.ashx` - Historical hourly data
2. **Local Weather** - `/weather.ashx` - Current + 14-day forecast (paid)
3. **Marine Weather** - `/marine.ashx` - Marine conditions

### Pricing
- **Free**: 500 calls/day - Past weather only
- **Premium**: $60/month - 10K calls/day + forecast
- **Enterprise**: Custom pricing

### Key Advantage: Historical Weather
**This is the only free API with good historical weather data.**

### Use Case: Weather Similarity Search

#### Problem
You want to project player performance in upcoming weather, but need historical context:
- "How did Patrick Mahomes perform in similar 20mph wind games?"
- "Find all games where weather matched today's forecast"

#### Solution: Weather Similarity Matching
1. **Fetch historical weather** for past games
2. **Calculate weather signatures** (wind/temp/precip scores)
3. **Find similar games** with matching signatures
4. **Analyze player stats** from those similar-weather games
5. **Adjust projections** based on historical patterns

### Weather Signature System

Each game gets a signature like `3_2_0`:
- **First digit**: Wind impact (0-4)
- **Second digit**: Cold impact (0-4)
- **Third digit**: Precip impact (0-3)

**Examples:**
- `0_0_0`: Perfect weather (or dome)
- `2_1_0`: Moderate wind, mild cold, no rain
- `4_4_2`: Extreme wind, extreme cold, heavy precip

### Similarity Search Query

```sql
-- Find games with weather signature close to target
WITH target AS (
  SELECT '3_2_0' as target_signature  -- Target weather
)
SELECT h.*, p.fantasy_points
FROM bronze_nfl_weather_historical h
JOIN gold_weekly_stats p 
  ON h.team = p.team 
  AND h.season = p.season 
  AND h.week = p.week
CROSS JOIN target
WHERE h.weather_signature = target.target_signature
  AND p.player_name = 'Patrick Mahomes'
```

### Integration with FantasAI Model

1. **Feature Engineering**: Add historical weather similarity features
2. **Model Training**: Train on games grouped by weather signature
3. **Prediction**: Use similar-weather historical performance

```python
# Find similar weather games for a player
def find_similar_weather_games(player, target_weather_signature):
    similar_games = spark.sql(f"""
        SELECT season, week, fantasy_points
        FROM gold_weekly_stats s
        JOIN bronze_nfl_weather_historical w
          ON s.team = w.team 
          AND s.season = w.season 
          AND s.week = w.week
        WHERE s.player_name = '{player}'
          AND w.weather_signature = '{target_weather_signature}'
    """)
    
    avg_performance = similar_games.agg({'fantasy_points': 'avg'}).collect()[0][0]
    return avg_performance
```

### Best Practices

1. **Build historical database**: Fetch past 2-3 seasons of game day weather
2. **Run once per week**: Backfill historical data, not real-time
3. **500 calls/day limit**: ~16 stadiums * 18 weeks = 288 calls per season
4. **Cache heavily**: Store all historical data in Delta tables
5. **Combine with player stats**: Join weather with fantasy points

### Rate Limit Management

```python
# 500 calls/day = ~20 per hour = 1 every 3 minutes
import time

for stadium, date in game_schedule:
    fetch_historical_weather(stadium, date)
    time.sleep(3.2)  # Stay under limit
```